# Prepare the four-pair evaluation

Run this notebook before the model notebooks. It lists the planned
experiments, prepares worksheets for later reviews and checks the supplied
input files. It does not train models or generate model outputs.

These steps support the assignment methodology: both models use the saved
tasks and inputs, and every final output has a place for your review.
The shared functions are in `a1`, so running and scoring use the same jobs.

Start the local kernel in the notebooks folder and install
requirements-analysis.txt in the CPU analysis environment. If a cloud
kernel starts elsewhere, edit project_root below. Inputs are licensed
public photographs and a mock retail catalogue, not IKEA operational data.


In [ ]:
# Inspect the fixed input selection and prepare missing review templates.
# Inputs are saved files; outputs are displays or derived CPU records.
# All inference and public-data download switches remain disabled.
from pathlib import Path
import sys
# Avoid creating interpreter caches in the submitted project.
sys.dont_write_bytecode = True

project_root = Path.cwd().resolve()
if not (project_root / "config/study.json").is_file():
    project_root = project_root.parent
if not (project_root / "config/study.json").is_file():
    raise FileNotFoundError("Start this notebook from the evaluation project")
sys.path.insert(0, str(project_root))


In [ ]:
# Read the planned jobs before running any models.
# A job is one model/input/seed combination. A pair compares models A and B.
from collections import Counter
from a1.study import planned_jobs
from a1.analysis import review_template
from a1.feedback import prepare_feedback

# planned_jobs reads config/study.json and config/models.json.
# Count how many final runs belong to each experiment (P01 to P04).
final_jobs = planned_jobs(project_root)
pair_counts = Counter()
for job in final_jobs:
    pair_counts[job["pair"]] += 1
print("Final jobs:", dict(pair_counts))

# Pilots check that models work; reproductions repeat selected pilots
# in fresh environments. These calls only describe the work.
pilot_jobs = planned_jobs(project_root, phase="pilot")
reproduction_jobs = planned_jobs(project_root, phase="reproduction")
print("Pilots:", len(pilot_jobs))
print("Reproductions:", len(reproduction_jobs))

# Make one review row per final job in data/manual_reviews.csv.
# Existing matching reviews are kept; filled retired rows require attention.
review_path = review_template(project_root)
print("Review worksheet:", review_path)

# Create the pair-level observations worksheet and guide only if missing.
# You must fill it after reviewing actual outputs; blank rows are not feedback.
feedback_template = prepare_feedback(project_root)
print(feedback_template)


In [ ]:
# Check the fixed source inputs used by the four experiments.
from a1.data import prepare_abo, validate_data

# Keep False when using the supplied files. True rebuilds prepared images
# and candidate masks, downloads missing originals and rewrites provenance.
# It reuses data/abo_selection.json; it does not choose different products.
DOWNLOAD_PUBLIC_DATA = False
if DOWNLOAD_PUBLIC_DATA:
    preparation_summary = prepare_abo(project_root)
    print(preparation_summary)

# Each check records a required path, whether it exists and its checksum.
# Repeated uses of one file are counted as separate required references.
checks = validate_data(project_root)
present_count = 0
for check in checks:
    if check["present"]:
        present_count += 1
print("Required files present:", present_count, "/", len(checks))


Expected counts for the saved configuration: P01 = 30, P02 = 12,
P03 = 12 and P04 = 24 final jobs; eight pilots and four reproductions.
These are planned counts, not completed experiments.

Before scoring, review the visible product labels, foreground masks and
mock-agent expected answers. File presence does not establish correctness.
The masks are candidates produced from near-white background pixels.
Preparation uses catalogue photographs and single-object still images;
no training or automatic product reselection is performed.
